# Name: Baraka Mandi

# Question 5: Implementing and Comparing CNN Architectures

This notebook implements the two architectures from the assignment in **PyTorch** (instead of TensorFlow/Keras):

1. **Task 1** – a simplified **AlexNet**
2. **Task 2** – a **Residual Block** and a small **ResNet-like** model

In [6]:
# Install torchinfo if it's not already available
# !pip install torchinfo

import torch
import torch.nn as nn
from torchinfo import summary

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")

PyTorch version: 2.13.0
Device: cpu


---
## Task 1: Simplified AlexNet

In [7]:
class AlexNet(nn.Module):
    """Simplified AlexNet for 10-class classification."""

    def __init__(self, in_channels: int = 3, num_classes: int = 10):
        super().__init__()
        self.features = nn.Sequential(
            # Conv2D: 96 filters, 11x11, stride 4, ReLU
            nn.Conv2d(in_channels, 96, kernel_size=11, stride=4),
            nn.ReLU(inplace=True),
            # MaxPool: 3x3, stride 2
            nn.MaxPool2d(kernel_size=3, stride=2),

            # Conv2D: 256 filters, 5x5, ReLU
            nn.Conv2d(96, 256, kernel_size=5, padding=2),
            nn.ReLU(inplace=True),
            # MaxPool: 3x3, stride 2
            nn.MaxPool2d(kernel_size=3, stride=2),

            # Conv2D: 384 filters, 3x3, ReLU
            nn.Conv2d(256, 384, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            # Conv2D: 384 filters, 3x3, ReLU
            nn.Conv2d(384, 384, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            # Conv2D: 256 filters, 3x3, ReLU
            nn.Conv2d(384, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            # MaxPool: 3x3, stride 2
            nn.MaxPool2d(kernel_size=3, stride=2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),                       # Flatten
            nn.Linear(256 * 6 * 6, 4096),       # Dense 4096, ReLU
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.5),                  # Dropout 50%
            nn.Linear(4096, 4096),              # Dense 4096, ReLU
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.5),                  # Dropout 50%
            nn.Linear(4096, num_classes),       # Output: 10 neurons
            nn.Softmax(dim=1),                  # Softmax activation
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


alexnet = AlexNet(num_classes=10).to(device)

### AlexNet model summary

In [8]:
alexnet_summary = summary(
    alexnet,
    input_size=(1, 3, 227, 227),   # (batch, channels, height, width)
    col_names=("input_size", "output_size", "num_params"),
    col_width=18,
    row_settings=("var_names",),
    device=device,
)
print(alexnet_summary)

Layer (type (var_name))                  Input Shape        Output Shape       Param #
AlexNet (AlexNet)                        [1, 3, 227, 227]   [1, 10]            --
├─Sequential (features)                  [1, 3, 227, 227]   [1, 256, 6, 6]     --
│    └─Conv2d (0)                        [1, 3, 227, 227]   [1, 96, 55, 55]    34,944
│    └─ReLU (1)                          [1, 96, 55, 55]    [1, 96, 55, 55]    --
│    └─MaxPool2d (2)                     [1, 96, 55, 55]    [1, 96, 27, 27]    --
│    └─Conv2d (3)                        [1, 96, 27, 27]    [1, 256, 27, 27]   614,656
│    └─ReLU (4)                          [1, 256, 27, 27]   [1, 256, 27, 27]   --
│    └─MaxPool2d (5)                     [1, 256, 27, 27]   [1, 256, 13, 13]   --
│    └─Conv2d (6)                        [1, 256, 13, 13]   [1, 384, 13, 13]   885,120
│    └─ReLU (7)                          [1, 384, 13, 13]   [1, 384, 13, 13]   --
│    └─Conv2d (8)                        [1, 384, 13, 13]   [1, 384, 13, 13]   

Quick sanity check

In [9]:
alexnet.eval()
with torch.no_grad():
    x = torch.randn(4, 3, 227, 227, device=device)
    out = alexnet(x)
print("Output shape:", tuple(out.shape))
print("Row sums (should be 1):", out.sum(dim=1).cpu().numpy().round(4))

Output shape: (4, 10)
Row sums (should be 1): [1. 1. 1. 1.]


---
## Task 2: Residual Block and ResNet-like Model

### 2.1 Residual block


In [10]:
class ResidualBlock(nn.Module):
    """Two 3x3 conv layers with an identity skip connection added before the final ReLU."""

    def __init__(self, in_channels: int = 64, filters: int = 64):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, filters, kernel_size=3, padding=1)
        self.relu1 = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(filters, filters, kernel_size=3, padding=1)
        self.relu_out = nn.ReLU(inplace=True)

        # Projection shortcut only needed if channel counts differ
        if in_channels != filters:
            self.shortcut = nn.Conv2d(in_channels, filters, kernel_size=1)
        else:
            self.shortcut = nn.Identity()

    def forward(self, input_tensor):
        shortcut = self.shortcut(input_tensor)
        x = self.relu1(self.conv1(input_tensor))  # Conv 1 + ReLU
        x = self.conv2(x)                          # Conv 2 (activation deferred)
        x = x + shortcut                           # Skip connection: add input
        return self.relu_out(x)                    # ReLU after the addition


def residual_block(input_tensor: torch.Tensor, filters: int = 64) -> torch.Tensor:
    """Functional-style helper matching the assignment's signature.

    Builds a ResidualBlock sized for `input_tensor` and applies it. The weights
    are freshly initialised on every call, so this is for demonstration only;
    inside a trainable model use the ResidualBlock module (as ResNetLike does).
    """
    block = ResidualBlock(in_channels=input_tensor.shape[1], filters=filters).to(input_tensor.device)
    return block(input_tensor)


# Demonstrate the functional helper: shape is preserved through the block
t = torch.randn(2, 64, 16, 16, device=device)
print("residual_block input shape :", tuple(t.shape))
print("residual_block output shape:", tuple(residual_block(t, filters=64).shape))

residual_block input shape : (2, 64, 16, 16)
residual_block output shape: (2, 64, 16, 16)


### 2.2 ResNet-like model



In [11]:
class ResNetLike(nn.Module):
    """Small ResNet-style network: stem conv -> 2 residual blocks -> dense head."""

    def __init__(self, in_channels: int = 3, num_classes: int = 10, input_hw: int = 32):
        super().__init__()
        # Initial Conv2D: 64 filters, 7x7, stride 2
        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=7, stride=2, padding=3),
            nn.ReLU(inplace=True),
        )
        # Two residual blocks
        self.res_block1 = ResidualBlock(64, 64)
        self.res_block2 = ResidualBlock(64, 64)

        feat_hw = (input_hw + 2 * 3 - 7) // 2 + 1   # spatial size after the stem
        # Flatten -> Dense(128) -> Output(Softmax)
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * feat_hw * feat_hw, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, num_classes),
            nn.Softmax(dim=1),
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.res_block1(x)
        x = self.res_block2(x)
        return self.head(x)


resnet = ResNetLike(num_classes=10, input_hw=32).to(device)

### ResNet-like model summary

In [12]:
resnet_summary = summary(
    resnet,
    input_size=(1, 3, 32, 32),
    col_names=("input_size", "output_size", "num_params"),
    col_width=18,
    row_settings=("var_names",),
    depth=3,
    device=device,
)
print(resnet_summary)

Layer (type (var_name))                  Input Shape        Output Shape       Param #
ResNetLike (ResNetLike)                  [1, 3, 32, 32]     [1, 10]            --
├─Sequential (stem)                      [1, 3, 32, 32]     [1, 64, 16, 16]    --
│    └─Conv2d (0)                        [1, 3, 32, 32]     [1, 64, 16, 16]    9,472
│    └─ReLU (1)                          [1, 64, 16, 16]    [1, 64, 16, 16]    --
├─ResidualBlock (res_block1)             [1, 64, 16, 16]    [1, 64, 16, 16]    --
│    └─Identity (shortcut)               [1, 64, 16, 16]    [1, 64, 16, 16]    --
│    └─Conv2d (conv1)                    [1, 64, 16, 16]    [1, 64, 16, 16]    36,928
│    └─ReLU (relu1)                      [1, 64, 16, 16]    [1, 64, 16, 16]    --
│    └─Conv2d (conv2)                    [1, 64, 16, 16]    [1, 64, 16, 16]    36,928
│    └─ReLU (relu_out)                   [1, 64, 16, 16]    [1, 64, 16, 16]    --
├─ResidualBlock (res_block2)             [1, 64, 16, 16]    [1, 64, 16, 16]    --
